# SceneSense Google Colab Launcher

Notebook ini berfungsi sebagai launcher untuk menjalankan pipeline training model **SceneSense** di Google Colab. 

### Prinsip & Desain Launcher:
- **Local-First & Single Codebase**: Seluruh business logic (preprocessing, training, kelanjutan training/resume, ekspor model, evaluasi, inferensi) dikendalikan sepenuhnya oleh source code proyek. Notebook ini hanya bertindak sebagai launcher/orkestrator eksekusi.
- **Idempotent**: Seluruh cell aman dijalankan berkali-kali tanpa melakukan duplikasi git clone, instalasi ulang package, unduhan dataset, atau split dataset berlebih.
- **Integrasi Google Drive**: Seluruh output penting disimpan langsung ke Google Drive agar aman dari pemutusan koneksi runtime (*runtime disconnection*).

## Langkah 1: Mount Google Drive

Hubungkan Google Colab ke Google Drive Anda untuk menaruh direktori keluaran training secara aman.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Langkah 2: Clone atau Update Repositori GitHub

Periksa apakah folder proyek sudah ada di lokal Colab VM:
- Jika belum ada: Melakukan clone repositori dari GitHub.
- Jika sudah ada: Menjalankan `git pull` untuk mendapatkan pembaruan kode terbaru.

In [ ]:
import os

#@title Konfigurasi Repositori Git
REPO_URL = "https://github.com/ImamAriadi2022/SceneSense.git" #@param {type:"string"}
PROJECT_NAME = "SceneSense"
TARGET_PATH = f"/content/{PROJECT_NAME}"

if not os.path.exists(TARGET_PATH):
    print(f"Cloning repository to {TARGET_PATH}...")
    !git clone {REPO_URL} {TARGET_PATH}
else:
    print(f"Repository already exists at {TARGET_PATH}. Pulling latest updates...")
    %cd {TARGET_PATH}
    !git pull

%cd {TARGET_PATH}

## Langkah 3: Instalasi Dependencies

Menginstal pustaka Python yang dideklarasikan di `requirements.txt`. Opsi `-q` digunakan agar log keluaran lebih rapi.

In [ ]:
print("Installing requirements...")
!pip install -q -r requirements.txt
print("All dependencies are ready!")

## Langkah 4: Set Environment Variables untuk Mengarahkan Path ke Google Drive

Mendefinisikan variabel lingkungan `SCENESENSE_*` untuk mengalihkan penyimpanan checkpoints, log CSV, TensorBoard, visualisasi, dan ekspor model (SavedModel, TFLite, TFJS) langsung ke direktori Google Drive Anda.

In [ ]:
import os

DRIVE_BASE = "/content/drive/MyDrive/SceneSense"

# Set variabel lingkungan
os.environ["SCENESENSE_MODEL_PATH"] = f"{DRIVE_BASE}/saved_model"
os.environ["SCENESENSE_CHECKPOINT_PATH"] = f"{DRIVE_BASE}/checkpoints"
os.environ["SCENESENSE_LOG_PATH"] = f"{DRIVE_BASE}/logs"
os.environ["SCENESENSE_TFLITE_PATH"] = f"{DRIVE_BASE}/tflite"
os.environ["SCENESENSE_TFJS_PATH"] = f"{DRIVE_BASE}/tfjs_model"
os.environ["SCENESENSE_OUTPUT_PATH"] = f"{DRIVE_BASE}/outputs"

# Memastikan folder tujuan di Google Drive sudah dibuat
for folder_path in [os.environ["SCENESENSE_MODEL_PATH"], 
                     os.environ["SCENESENSE_CHECKPOINT_PATH"], 
                     os.environ["SCENESENSE_LOG_PATH"], 
                     os.environ["SCENESENSE_TFLITE_PATH"], 
                     os.environ["SCENESENSE_TFJS_PATH"], 
                     os.environ["SCENESENSE_OUTPUT_PATH"]]:
    os.makedirs(folder_path, exist_ok=True)

print("Environment variables set and output directories created in Google Drive!")

## Langkah 5: Unduh Dataset Mentah

Menjalankan script `download_dataset.py` untuk mengunduh dataset gambar Intel dari Kaggle secara otomatis. Jika dataset sudah diunduh sebelumnya, script akan langsung mendeteksi dan melewati proses ini secara idempotent.

In [ ]:
!python download_dataset.py

## Langkah 6: Validasi Dataset

Memeriksa apakah dataset gambar berhasil diunduh dan diintegrasikan ke folder raw lokal VM.

In [ ]:
import os

raw_dir = "dataset/raw"
if os.path.isdir(raw_dir) and len(os.listdir(raw_dir)) > 0:
    print("✅ Dataset validation successful!")
    print(f"Classes found: {os.listdir(raw_dir)}")
    total_images = sum(len(files) for _, _, files in os.walk(raw_dir))
    print(f"Total raw images: {total_images}")
else:
    print("❌ Dataset validation failed. Please check the download steps.")

## Langkah 7: Eksekusi Proses Training

Menjalankan program utama `train.py`. 
- Jika dijalankan pertama kali: Dataset mentah akan di-split secara otomatis, model dibuat baru, dan melatih dari epoch 0.
- Jika dijalankan ulang setelah terputus: Pembagian dataset akan dilewati (karena direktori splits sudah ada), dan training secara otomatis melanjutkan (*resume*) dari epoch terakhir yang terekam pada berkas log Google Drive.

In [ ]:
!python train.py --data_dir dataset/raw

## Langkah 8: Evaluasi & Uji Prediksi Uji (Inference)

Mengambil salah satu berkas gambar tes secara acak dan menjalankan script `inference.py` untuk menguji SavedModel dan TFLite yang tersimpan di Google Drive.

In [ ]:
import random
import glob

test_images = glob.glob("dataset/test/*/*.jpg") + glob.glob("dataset/test/*/*.png")

if test_images:
    sample_img = random.choice(test_images)
    print(f"Selected sample image for inference: {sample_img}")
    
    # Jalankan inferensi menggunakan SavedModel
    print("\n--- Inference with SavedModel ---")
    model_dir = os.environ["SCENESENSE_MODEL_PATH"]
    !python inference.py --model_path "{model_dir}" --image_path "{sample_img}"
    
    # Jalankan inferensi menggunakan TFLite
    tflite_dir = os.environ["SCENESENSE_TFLITE_PATH"]
    tflite_file = os.path.join(tflite_dir, "model.tflite")
    if os.path.exists(tflite_file):
        print("\n--- Inference with TFLite model ---")
        !python inference.py --tflite_path "{tflite_file}" --image_path "{sample_img}"
else:
    print("No test images found. Make sure training ran at least once to split the dataset.")

---

## Ringkasan Lokasi Penyimpanan Data (Summary)

Setelah pelatihan berhasil atau di-resume, seluruh aset penting Anda disimpan pada folder Google Drive berikut:

| Tipe Aset | Lokasi Penyimpanan di Google Drive | Deskripsi |
| :--- | :--- | :--- |
| **Dataset** | `/content/SceneSense/dataset` (Lokal VM) | Menggunakan lokal VM disk Colab agar pembacaan batch I/O berjalan cepat. |
| **Checkpoints** | `/content/drive/MyDrive/SceneSense/checkpoints/` | Berisi `best_model.keras` (model terbaik) dan `latest_checkpoint.keras` (checkpoint terakhir). |
| **SavedModel** | `/content/drive/MyDrive/SceneSense/saved_model/` | Hasil ekspor TensorFlow SavedModel akhir dan berkas `labels.txt`. |
| **TFJS Model** | `/content/drive/MyDrive/SceneSense/tfjs_model/` | Model hasil konversi untuk pustaka web TensorFlow.js. |
| **TFLite Model** | `/content/drive/MyDrive/SceneSense/tflite/` | Berkas model terkuantisasi `model.tflite` untuk perangkat mobile. |
| **Logs & Grafik** | `/content/drive/MyDrive/SceneSense/logs/` & `/content/drive/MyDrive/SceneSense/outputs/` | Berisi data logging TensorBoard, riwayat log CSV, grafik performa, dan classification report. |